# Caso 1: Transitorio Térmico en un Motor de Inducción
Este cuaderno interactivo pertenece al libro *Métodos Numéricos en la Ingeniería Electromecánica*. 
Presiona el botón de **Play (▷)** a la izquierda de la celda de código inferior para ejecutar la simulación y generar los gráficos de análisis de error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuración de estilo académico "Nivel PhD"
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "figure.dpi": 150
})

def resolver_transitorio_termico(dt: float, t_final: float) -> tuple[np.ndarray, np.ndarray]:
    T_amb = 25.0        
    P_perdidas = 1200.0 
    C_th = 45000.0      
    tau = 1200.0        
    
    t = np.arange(0, t_final + dt, dt)
    T = np.zeros(len(t))
    T[0] = T_amb 
    
    for i in range(0, len(t) - 1):
        dT_dt = (P_perdidas / C_th) - (1.0 / tau) * (T[i] - T_amb)
        T[i+1] = T[i] + dT_dt * dt
        
    return t, T

t_final_sim = 3600.0  
pasos_tiempo = [120.0, 60.0, 10.0]

T_amb, P_perdidas, C_th, tau = 25.0, 1200.0, 45000.0, 1200.0
t_exacto = np.linspace(0, t_final_sim, 1000)
T_exacto = T_amb + (tau * P_perdidas / C_th) * (1.0 - np.exp(-t_exacto / tau))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(t_exacto / 60.0, T_exacto, label="Solución Exacta (Analítica)", color="black", linewidth=2.0)

estilos_linea = [('--', 'o'), (':', 's'), ('-.', '^')]
for dt, (estilo, marcador) in zip(pasos_tiempo, estilos_linea):
    t_num, T_num = resolver_transitorio_termico(dt, t_final_sim)
    ax1.plot(t_num / 60.0, T_num, linestyle=estilo, label=f"Euler ($\\Delta t = {dt}$ s)", alpha=0.85)
    T_exacta_coincidente = T_amb + (tau * P_perdidas / C_th) * (1.0 - np.exp(-t_num / tau))
    error_abs = np.abs(T_exacta_coincidente - T_num)
    ax2.plot(t_num / 60.0, error_abs, linestyle=estilo, label=f"Error ($\\Delta t = {dt}$ s)")

ax1.set_title("Evolución de la Temperatura")
ax1.set_xlabel("Tiempo (minutos)")
ax1.set_ylabel("Temperatura ($^{\\circ}$C)")
ax1.grid(True)
ax1.legend(loc="lower right")

ax2.set_title("Error de Truncamiento Global")
ax2.set_xlabel("Tiempo (minutos)")
ax2.set_ylabel("Error Absoluto ($^{\\circ}$C)")
ax2.grid(True)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()